# 03 · H1 и H2: различает ли модель ответы и пользуется ли этим

Без вмешательства. Отвечает на развилку проекта:

| H1 (probe) | H2 (поведение) | Что значит |
|---|---|---|
| ✔ | ошибается | **целевой разрыв**: различает, но не действует → steering осмыслен |
| ✔ | почти не ошибается | ошибок мало → менять постановку задач |
| ✘ | — | ответ не читается из активаций → steering вряд ли поможет; проверять промпт / явное состояние |

> **Протокол** (`docs/HYPOTHESES.md`, `AGENTS.md`): вектор строится только на train,
> подбор — только на val, test открывается **один раз** на замороженный метод.
> Пайплайн это проверяет сам — если ячейка упала с `TestLockError` или `StageOrderError`,
> это защита сработала, а не баг.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="h1h2-qwen3-1.7b", hypothesis="H2", notebook="notebooks/03_h1_h2_baseline.ipynb",
    notes="Первый замер: есть ли разрыв «различает, но не действует» на реальной модели.",
    overrides={"model.name": "Qwen/Qwen3-1.7B", "data.n_tasks": 160},
)
exp = Experiment(cfg)
exp.data()
exp.estimate()

In [ ]:
exp.baseline()
exp.results_table("val")

### Где ошибается база: по типам пар, доменам, явности условия, позиции

In [ ]:
recs = pd.DataFrame(exp.state["records"]["val"]["baseline"] + exp.state["records"]["train"]["baseline"])
recs.groupby(["pair_type", "domain", "explicit"])["ok"].agg(["mean", "count"])

In [ ]:
exp.probe()
viz.probe_heatmap(exp.probe_table());

In [ ]:
for h, v in exp.verdicts().items():
    print(f"{h}: {v.label:16} {v.reason}")

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))